# 🛡️ PPE Kit Detection & Live Alert Model Training
### High-Precision Pipeline using YOLOv8 & Roboflow Universe

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/new)

This notebook trains a state-of-the-art **YOLOv8** model on the clean **Roboflow Construction Site Safety Dataset**.

#### Clean Target Classes:
- 🪖 **Hardhat**
- 🦺 **Safety Vest**
- 😷 **Mask**
- 👤 **Person**
- 🚨 **NO-Hardhat** *(Direct Violation)*
- 🚨 **NO-Safety Vest** *(Direct Violation)*
- 🚨 **NO-Mask** *(Direct Violation)*

> **GPU Setup Reminder**:
> Click `Runtime` -> `Change runtime type` -> select `T4 GPU` -> `Save`.

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

## 2. Install Dependencies

In [ ]:
!pip install -q "ultralytics>=8.1.0" "roboflow" "opencv-python-headless" "matplotlib" "pyyaml"

In [ ]:
import torch
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import os
import glob

print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 3. Download Clean Roboflow PPE Dataset
We use the verified **Construction Site Safety (v28)** dataset from Roboflow Universe.

In [ ]:
import os
import glob
from pathlib import Path

# Enter your Roboflow API key below or leave blank for direct public download:
ROBOFLOW_API_KEY = ""  # <-- Optional: Paste your Roboflow API key here

data_yaml_path = None

if ROBOFLOW_API_KEY.strip():
    from roboflow import Roboflow
    print("📥 Downloading official Roboflow Construction Site Safety dataset...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY.strip())
    project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
    dataset = project.version(28).download("yolov8")
    data_yaml_path = f"{dataset.location}/data.yaml"
    print(f"✅ Downloaded via Roboflow API: {data_yaml_path}")
else:
    print("📥 Downloading public Roboflow Construction PPE Dataset...")
    dataset_dir = "/content/datasets/roboflow_ppe"
    os.makedirs(dataset_dir, exist_ok=True)
    
    !curl -L -o /content/roboflow_ppe.zip "https://universe.roboflow.com/ds/kX3bVvP7yC?key=hVzU2eZ4Wz"
    !unzip -q -o /content/roboflow_ppe.zip -d /content/datasets/roboflow_ppe/
    
    found_yamls = glob.glob("/content/**/data.yaml", recursive=True)
    data_yaml_path = found_yamls[0] if found_yamls else "/content/datasets/roboflow_ppe/data.yaml"

print(f"📄 Active data configuration: {data_yaml_path}")

## 4. Train YOLOv8 with GPU VRAM Optimization

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8s
model = YOLO('yolov8s.pt')

# Train with hardware-optimized parameters
results = model.train(
    data=data_yaml_path,
    epochs=40,
    imgsz=640,
    batch=32,
    cache=True,
    workers=4,
    device=0,
    project='ppe_detection_project',
    name='ppe_roboflow_yolov8s',
    patience=12,
    save=True,
    plots=True,
    verbose=True
)

print("🎉 Training Finished!")

## 5. Evaluate Model & Confusion Matrix

In [ ]:
# Automatically locate best.pt across all run folders
candidate_weights = glob.glob("/content/**/ppe_roboflow_yolov8s/**/best.pt", recursive=True)
if not candidate_weights:
    candidate_weights = glob.glob("/content/**/best.pt", recursive=True)

if candidate_weights:
    best_weights = candidate_weights[-1]
    print(f"✅ Evaluating best weights: {best_weights}")
    
    eval_model = YOLO(best_weights)
    metrics = eval_model.val()

    print(f"\n🏆 Final Validation mAP@50:    {metrics.box.map50:.4f}")
    print(f"🏆 Final Validation mAP@50-95: {metrics.box.map:.4f}")

    # Display Confusion Matrix
    save_dir = os.path.dirname(os.path.dirname(best_weights))
    cm_path = os.path.join(save_dir, "confusion_matrix.png")
    if os.path.exists(cm_path):
        img = cv2.imread(cm_path)
        plt.figure(figsize=(10, 8))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title('Clean Confusion Matrix - Roboflow PPE Detection')
        plt.show()
else:
    print("❌ best.pt not found yet. Please make sure training has completed!")

## 6. Test on Validation Images & Display Output Photos in Notebook
Picks sample images from your **validation dataset**, runs detection, prints compliance violations, and displays the annotated photos directly in the notebook!

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Load trained best model
candidate_weights = glob.glob("/content/**/best.pt", recursive=True)
best_path = candidate_weights[-1] if candidate_weights else 'yolov8s.pt'
ppe_model = YOLO(best_path)

# 2. Find images from the validation dataset
val_images = glob.glob("/content/**/val/images/*.jpg", recursive=True) or \
             glob.glob("/content/**/valid/images/*.jpg", recursive=True) or \
             glob.glob("/content/**/test/images/*.jpg", recursive=True)

if not val_images:
    print("❌ No validation images found. Please verify dataset download.")
else:
    print(f"✅ Found {len(val_images)} validation images. Testing on sample photos...\n")
    
    # Test on up to 3 sample photos from validation data
    sample_photos = val_images[:3]
    
    for idx, photo_path in enumerate(sample_photos):
        print("=" * 60)
        print(f"📸 Testing Validation Photo #{idx + 1}: {os.path.basename(photo_path)}")
        
        # Run inference
        preds = ppe_model.predict(source=photo_path, conf=0.40, verbose=False)
        r = preds[0]
        
        # Check detected items and violation rules
        detected_items = [r.names[c] for c in r.boxes.cls.cpu().numpy().astype(int)]
        print(f"📋 Detected Objects: {detected_items if detected_items else 'None'}")
        
        has_hardhat = any('hardhat' in x.lower() or 'helmet' in x.lower() for x in detected_items)
        has_vest = any('vest' in x.lower() and 'no' not in x.lower() for x in detected_items)
        has_no_hardhat = any('no-hardhat' in x.lower() or 'no-helmet' in x.lower() for x in detected_items)
        has_no_vest = any('no-safety vest' in x.lower() or 'no-vest' in x.lower() for x in detected_items)
        
        violations = []
        if has_no_hardhat or not has_hardhat:
            violations.append("MISSING HARDHAT")
        if has_no_vest or not has_vest:
            violations.append("MISSING SAFETY VEST")
            
        if violations:
            print(f"🚨 [ALERT] VIOLATIONS: {', '.join(violations)}")
        else:
            print("🟢 STATUS: FULLY COMPLIANT (Hardhat & Vest OK)")
            
        # Render and display the photo in notebook output
        annotated_bgr = r.plot()
        annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(10, 7))
        plt.imshow(annotated_rgb)
        plt.axis('off')
        title_text = f"Validation Photo #{idx+1}: {', '.join(violations) if violations else 'COMPLIANT'}"
        plt.title(title_text, fontsize=13, color='red' if violations else 'green', weight='bold')
        plt.show()
        print("=" * 60 + "\n")

## 7. Download Trained `best.pt` to Local PC
Downloads `best_ppe_yolov8.pt` to your computer. Drag & drop it into your local website!

In [ ]:
from google.colab import files
import shutil

candidate_weights = glob.glob("/content/**/ppe_roboflow_yolov8s/**/best.pt", recursive=True)
if not candidate_weights:
    candidate_weights = glob.glob("/content/**/best.pt", recursive=True)

if candidate_weights:
    best_weights = candidate_weights[-1]
    target_file = "/content/best_ppe_yolov8.pt"
    shutil.copy(best_weights, target_file)
    print(f"⬇️ Downloading {best_weights} as best_ppe_yolov8.pt...")
    files.download(target_file)
else:
    print("❌ Weights file not found yet.")